In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator , TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import  cross_validate ,train_test_split ,StratifiedKFold, RandomizedSearchCV , KFold

from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

from imblearn.over_sampling import SMOTE
from dataclasses import dataclass
import joblib
from sklearn.svm import SVC
import os
from sklearn.metrics import roc_auc_score

import scipy

In [2]:
linux_path = r"/run/media/drdrakken/Elements/Sonstiges/Programmieren/Machine Learning/csvs/driving/porto-seguro-safe-driver-prediction/train.csv"
df = pd.read_csv(linux_path)

In [3]:
df

,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
0,7,0,2,2,5,1,0,0,1,0,...,9,1,5,8,0,1,1,0,0,1
1,9,0,1,1,7,0,0,0,0,1,...,3,1,1,9,0,1,1,0,1,0
2,13,0,5,4,9,1,0,0,0,1,...,4,2,7,7,0,1,1,0,1,0
3,16,0,0,1,2,0,0,1,0,0,...,2,2,4,9,0,0,0,0,0,0
4,17,0,0,2,0,1,0,1,0,0,...,3,1,1,3,0,0,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595207,1488013,0,3,1,10,0,0,0,0,0,...,4,1,9,6,0,1,1,0,1,1
595208,1488016,0,5,1,3,0,0,0,0,0,...,4,1,3,8,1,0,1,0,1,1
595209,1488017,0,1,1,10,0,0,1,0,0,...,3,2,2,6,0,0,1,0,0,0
595210,1488021,0,5,2,3,1,0,0,0,1,...,4,1,4,2,0,1,1,1,0,0


In [4]:
@dataclass
class Config:
    target: str = "target"
    seed: int = 1234
    test_size: float = 0.2
    cross_iterations: int = 5
    save_path : str = r"/home/drdrakken/Downloads/ModelSaveDir/save_driver-ptk"
    verbose:int = 1234
    grid_train_amount:int = 20
config = Config()

In [5]:
class DataClass():
    def __init__(self , DataFrame = df):
        self.x = DataFrame.drop([config.target] , axis = 1)
        self.y = DataFrame[config.target]

        self.numerical_data = self.x.select_dtypes(include = np.number).columns
        self.categorical_data = self.x.select_dtypes(exclude = np.number).columns
data = DataClass()            

In [6]:
class Visualize():

    def __init__(self):
        self.data = data.x

    def iqr(self, TargetCol):
        q1 = self.data[TargetCol].quantile(0.25)
        q3 = self.data[TargetCol].quantile(0.75)
        iqr = q3 - q1
        return q1 , q3 , iqr
    
    def skew(self , SkewCol):
        data_skew = self.data[SkewCol].skew()
        print(f"Skewness of Col:{SkewCol}->:{data_skew}")

    def plot(self):
        for vis in self.data.numerical_data:
            fig , axes = plt.subplots(3 , 1 , figsize = (10 , 10) , dpi = 200)
            q1 , q3 , iqr = self.iqr(TargetCol = vis)
            mean = self.data[vis].mean()
            self.skew(SkewCol = vis)

            sns.histplot(data = self.data , x = vis , ax = axes[0])
            axes[0].axvline(q1 , color = "green")
            axes[0].axvline(q3 , color = "red")
            axes[0].axvline(mean , color = "yellow")
            axes[0].set_title(f"Histplot for:{vis}")

            sns.boxplot(data = self.data , x = vis , ax = axes[1])
            axes[1].axvline(q1 , color = "green")
            axes[1].axvline(q3 , color = "red")
            axes[1].axvline(mean , color = "yellow")
            axes[1].set_title(f"Boxplot for:{vis}")

            sns.scatterplot(data = self.data , x = vis , ax = axes[2])
            axes[2].axvline(q1 , color = "green")
            axes[2].axvline(q3 , color = "red")
            axes[2].axvline(mean , color = "yellow")
            axes[2].set_title(f"Scatterplot for:{vis}")

            plt.tight_layout()
            plt.show()


In [7]:
class DataPreprcocess(BaseEstimator, TransformerMixin):

    def fit(self , X , y = None):
        numerical_cols = X.select_dtypes(include = np.number).columns
        categorical_cols = X.select_dtypes(exclude = np.number).columns

        self.preprocess = ColumnTransformer([
            ("numerical_data_process" , Pipeline([
                ("imputer" , SimpleImputer(strategy = "mean")),
                ("scale" , MinMaxScaler()),
            ]),numerical_cols),

            ("categorical_data_process" , Pipeline([
                ("imputer" , SimpleImputer(strategy = "most_frequent")),
                ("encoder" , OneHotEncoder(handle_unknown = "ignore" , sparse_output=False)),
            ]),categorical_cols),
        ])
        self.preprocess.fit(X)
        return self
    
    def transform(self , X , y = None):
        return self.preprocess.transform(X)

In [8]:
class DataTransform(BaseEstimator , TransformerMixin):

    def fit(self , X , y = None):
        return self
    
    def transform(self, X , y = None):
        return self.new_features(X)
    
    def new_features(self , X , y = None):
        x = X.copy()

        return x
    
    def combine_features(self , X):
        x = X.copy()
        #combined = [comb for comb in x.columns if comb startswith("ps_ind")]
        return x

In [9]:
def model_varianz():
    return {

        "LogisticRegression":LogisticRegression(random_state = config.seed , max_iter = 1000),
        "SVC":SVC(random_state = config.seed , kernel = "linear" , probability = True ),
        "RandomForestClassifier":RandomForestClassifier(random_state = config.seed),
        "DecisionTreeClassifier":DecisionTreeClassifier(random_state = config.seed),
        
    }

In [10]:
def custom_scoring_dict():

    return {

        "f1":"f1",
        "precision":"precision",
        "accuracy":"accuracy"
    }

In [11]:
def custom_model_parameters():
    return {

        "LogisticRegression": {
            "estimator__C": scipy.stats.loguniform(1e-4, 1e2),
            "estimator__solver": ["lbfgs"],
        },

        "DecisionTreeClassifier": {
            "estimator__criterion": ["gini", "entropy"],
            "estimator__max_depth": [None, 5, 10, 20],
            "estimator__min_samples_split": scipy.stats.randint(2, 20),
            "estimator__min_samples_leaf": scipy.stats.randint(1, 10),
        },

        "RandomForestClassifier": {
            "estimator__n_estimators": scipy.stats.randint(100, 500),
            "estimator__max_depth": scipy.stats.randint(5, 40),
            "estimator__min_samples_split": scipy.stats.randint(2, 20),
            "estimator__min_samples_leaf": scipy.stats.randint(1, 10),
            "estimator__max_features": ["sqrt", "log2"],
            "estimator__bootstrap": [True, False],
        },

        "LinearSVC": {
            "estimator__C": scipy.stats.loguniform(1e-4, 1e2),
            "estimator__loss": ["hinge", "squared_hinge"],
            "estimator__dual": [True],
            "estimator__max_iter": [5000, 10000],
        },

        "XGBClassifier": {
            "estimator__n_estimators": scipy.stats.randint(100, 500),
            "estimator__learning_rate": scipy.stats.loguniform(1e-3, 0.3),
            "estimator__max_depth": scipy.stats.randint(3, 10),
            "estimator__subsample": scipy.stats.uniform(0.6, 0.4),
            "estimator__colsample_bytree": scipy.stats.uniform(0.6, 0.4),
            "estimator__gamma": scipy.stats.uniform(0, 5),
            "estimator__min_child_weight": scipy.stats.randint(1, 10),
        },

    }


In [12]:
def data_split():
    X_train , X_test , y_train , y_test = train_test_split(data.x,
                                                           data.y,
                                                           shuffle = True,
                                                           random_state = config.seed,
                                                           test_size = config.test_size,
                                                           stratify = data.y)
    
    return  X_train , X_test , y_train , y_test

In [13]:
def custom_data_validation(estimator , X_train , y_train):
    kfold = StratifiedKFold(n_splits = config.cross_iterations , shuffle = True , random_state = config.seed)

    return cross_validate(estimator = estimator,
                          X= X_train,
                          y= y_train,
                          return_train_score = True,
                          scoring="roc_auc",
                          cv = kfold,
                          return_estimator=True,
                          verbose=config.verbose,
                          )


In [14]:
def custom_pipeline(estimator , transform = False , smote = False):
    steps = []

    steps.append(("BasePeformance" , DataPreprcocess()))
    if transform:
        steps.append(("TransformPeformance" , DataTransform()))

    if smote:
        steps.append(("smote" , SMOTE()))
        
    steps.append(("estimator" , estimator))
    return Pipeline(steps)

In [15]:
def custom_grid_search(estimator , param_grid , scoring , X_train , y_train):
    k_fold = KFold(n_splits= 5 , shuffle= True , random_state=config.seed)
    
    grid = RandomizedSearchCV(estimator=estimator,
                              param_distributions= param_grid ,
                              n_iter=config.grid_train_amount,
                              scoring = scoring,
                              n_jobs=-1,
                              return_train_score=True,
                              cv = k_fold,
                              refit="f1")
    
    grid.fit(X_train , y_train)
    return grid

In [16]:
def plot_scores(
        self,
        data,
        x_column,
        score_columns,
        title="Model Comparison",
        ylabel="Score"
    ):
        # Falls eine Liste von Dictionaries übergeben wird
        if isinstance(data, list):
            data = pd.DataFrame(data)

        x = range(len(data))

        plt.figure(figsize=(12, 6))

        for column in score_columns:
            if column in data.columns:
                plt.plot(
                    x,
                    data[column],
                    marker="o",
                    linewidth=2,
                    label=column
                )

        plt.xticks(x, data[x_column], rotation=45)
        plt.xlabel(x_column)
        plt.ylabel(ylabel)
        plt.title(title)
        plt.grid(True, linestyle="--", alpha=0.5)
        plt.legend()
        plt.tight_layout()
        plt.show()

In [17]:
class Benchmark():

    def __init__(self , UseCV = False , UseGrid = False ):
        self.X_train , self.X_test , self.y_train , self.y_test = data_split()
        self.results = []

        self.use_cv = UseCV
        self.use_grid_search = UseGrid
        self.train()

    def train(self):

        for estimator_name , estimators in model_varianz().items():

            base_pipe = custom_pipeline(estimator=estimators , transform= False , smote=False)
            transformed_pipe = custom_pipeline(estimator=estimators , transform= True , smote=True)

            if self.use_cv:
                base_cv = custom_data_validation(estimator=base_pipe,
                                                X_train=self.X_train,
                                                y_train=self.y_train)
                
                transformed_cv = custom_data_validation(estimator=transformed_pipe,
                                                X_train=self.X_train,
                                                y_train=self.y_train)
                print(f"keys;{base_cv.keys()}")

                self.results.append({

                    "base_cv_train_performance":base_cv["train_score"].mean(),
                    "base_cv_test_performance":base_cv["test_score"].mean(),
                    "base_cv_std_performance":base_cv["train_score"].std(),

                    "transformed_cv_train_performance":transformed_cv["train_score"].mean(),
                    "transformed_cv_train_performance":transformed_cv["train_score"].mean(),
                    "transformed_cv_std_performance":transformed_cv["train_score"].std(),
                })

            if self.use_grid_search:
                base_grid = custom_grid_search(estimator=base_pipe,
                                            scoring= custom_scoring_dict(),
                                            X_train=self.X_train,
                                            y_train=self.y_train,
                                            param_grid=custom_model_parameters()[estimator_name])

                transformed_grid = custom_grid_search(estimator=base_pipe,
                                            scoring= custom_scoring_dict(),
                                            X_train=self.X_train,
                                            y_train=self.y_train,
                                            param_grid=custom_model_parameters()[estimator_name])
                
            self.results.append({

                "base_cv_train_performance":base_grid.best_estimator_,
                "transformed_cv_train_performance":transformed_grid.best_estimator_,
                
            })

        plot_scores(data = self.results )
 

In [ ]:
Benchmark(UseGrid=True , UseCV=True)

[CV] START .....................................................................
[CV] END ..................., score=(train=0.623, test=0.624) total time=   9.3s
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:   10.2s
[CV] START .....................................................................
[CV] END ..................., score=(train=0.624, test=0.614) total time=   7.9s
[Parallel(n_jobs=1)]: Done   2 tasks      | elapsed:   18.9s
[CV] START .....................................................................
[CV] END ..................., score=(train=0.622, test=0.620) total time=   6.9s
[Parallel(n_jobs=1)]: Done   3 tasks      | elapsed:   26.5s
[CV] START .....................................................................
[CV] END ..................., score=(train=0.624, test=0.612) total time=   6.3s
[Parallel(n_jobs=1)]: Done   4 tasks      | elapsed:   33.4s
[CV] START .....................................................................
[CV] END ..................

/home/drdrakken/.clean_venv/lib64/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/drdrakken/.clean_venv/lib64/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/drdrakken/.clean_venv/lib64/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/hom

[CV] START .....................................................................


/home/drdrakken/.clean_venv/lib64/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


[CV] END .................., score=(train=0.513, test=0.519) total time=160.9min
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed: 168.5min
[CV] START .....................................................................


/home/drdrakken/.clean_venv/lib64/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
